In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
import os
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
# load everything
DATA_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'v3' else os.getcwd()
xlsx = pd.ExcelFile(os.path.join(DATA_DIR, 'Data for Datathon (Revised).xlsx'))
template = pd.read_csv(os.path.join(DATA_DIR, 'template_forecast_v00.csv'))

portfolios = ['A', 'B', 'C', 'D']

daily = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Daily')
    df['Date'] = pd.to_datetime(df['Date'].str.strip().str.rsplit(' ', n=1).str[0], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)
    df.columns = [c.strip() for c in df.columns]
    daily[p] = df

print({p: len(daily[p]) for p in portfolios})
print(daily['A'].columns.tolist())
daily['A'].head(2)


{'A': 731, 'B': 731, 'C': 731, 'D': 731}
['Date', 'Call Volume', 'CCT', 'Service Level', 'Abandon Rate']


,Date,Call Volume,CCT,Service Level,Abandon Rate
0,2024-01-01,2147.0,302.45,0.9855,0.0037
1,2024-01-02,7458.0,349.22,0.5213,0.1136


In [3]:
# interval data - only have apr-jun 2024 which kinda sucks
mmap = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
        'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}

intervals = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Interval')
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(subset=['Interval']).copy()
    df['mnum'] = df['Month'].map(mmap)
    df['Day'] = df['Day'].astype(int)
    df['Date'] = pd.to_datetime(dict(year=2024, month=df['mnum'], day=df['Day']))
    df['slot'] = df['Interval'].apply(lambda t: t.hour * 2 + t.minute // 30)
    df = df.sort_values(['Date','slot']).reset_index(drop=True)
    intervals[p] = df

print({p: len(intervals[p]) for p in portfolios})
# print(intervals['A']['slot'].unique())


{'A': 4076, 'B': 4285, 'C': 4359, 'D': 4358}


In [4]:
# staffing
staff = pd.read_excel(xlsx, 'Daily Staffing')
staff.columns = ['Date'] + [f'Staff_{p}' for p in portfolios]
staff['Date'] = pd.to_datetime(staff['Date'])
staff = staff.sort_values('Date').reset_index(drop=True)
len(staff)


365

In [5]:
# idea: more calls per agent = people wait longer = more abandon
# lets check if thats actually true
staff_lr = {}
for p in portfolios:
    tmp = daily[p].merge(staff[['Date', f'Staff_{p}']], on='Date', how='left')
    tmp = tmp.dropna(subset=[f'Staff_{p}', 'Call Volume', 'Abandon Rate', 'CCT'])
    tmp['cpa'] = tmp['Call Volume'] / tmp[f'Staff_{p}']
    X = tmp['cpa'].values.reshape(-1,1)
    staff_lr[p] = {
        'abd': LinearRegression().fit(X, tmp['Abandon Rate'].values),
        'cct': LinearRegression().fit(X, tmp['CCT'].values)
    }
    r = tmp['cpa'].corr(tmp['Abandon Rate'])
    print(f'{p}: corr={r:.3f}', '(weak)' if abs(r)<0.2 else '(decent)')
# not super strong but worth using


A: corr=0.155 (weak)
B: corr=-0.101 (weak)
C: corr=0.146 (weak)
D: corr=0.161 (weak)


In [6]:
# feature engineering
# tried a bunch of things, these seemed to help the most

holidays = pd.to_datetime(['2024-01-01','2024-01-15','2024-02-19','2024-05-27','2024-06-19',
    '2024-07-04','2024-09-02','2024-10-14','2024-11-11','2024-11-28','2024-12-25',
    '2025-01-01','2025-01-20','2025-02-17','2025-05-26','2025-06-19',
    '2025-07-04','2025-09-01','2025-10-13','2025-11-11','2025-11-27','2025-12-25'])

def make_features(df, port):
    f = pd.DataFrame()
    f['Date'] = df['Date']
    f['dow'] = df['Date'].dt.dayofweek
    f['dom'] = df['Date'].dt.day
    f['month'] = df['Date'].dt.month
    f['woy'] = df['Date'].dt.isocalendar().week.astype(int)
    f['qtr'] = df['Date'].dt.quarter
    f['year'] = df['Date'].dt.year
    f['wknd'] = (f['dow'] >= 5).astype(int)
    f['mon'] = (f['dow'] == 0).astype(int)  # mondays are always crazy
    f['fri'] = (f['dow'] == 4).astype(int)
    
    # cyclical encoding
    f['dow_s'] = np.sin(2*np.pi*f['dow']/7)
    f['dow_c'] = np.cos(2*np.pi*f['dow']/7)
    f['dom_s'] = np.sin(2*np.pi*f['dom']/31)
    f['dom_c'] = np.cos(2*np.pi*f['dom']/31)
    f['mon_s'] = np.sin(2*np.pi*f['month']/12)
    f['mon_c'] = np.cos(2*np.pi*f['month']/12)
    
    f['holiday'] = df['Date'].isin(holidays).astype(int)
    harr = holidays.values
    ds, du = [], []
    for d in df['Date'].values:
        past = harr[harr <= d]
        fut = harr[harr >= d]
        ds.append((d-past[-1])/np.timedelta64(1,'D') if len(past)>0 else 365)
        du.append((fut[0]-d)/np.timedelta64(1,'D') if len(fut)>0 else 365)
    f['d_since_hol'] = ds
    f['d_until_hol'] = du
    
    # billing cycle - people call more beginning/end of month
    f['month_start'] = (f['dom'] <= 5).astype(int)
    f['month_end'] = (f['dom'] >= 26).astype(int)
    f['first'] = (f['dom'] == 1).astype(int)
    
    # fourier terms
    didx = (df['Date'] - df['Date'].min()).dt.days
    for k in range(1,4):
        f[f'ws{k}'] = np.sin(2*np.pi*k*didx/7)
        f[f'wc{k}'] = np.cos(2*np.pi*k*didx/7)
        f[f'ys{k}'] = np.sin(2*np.pi*k*didx/365.25)
        f[f'yc{k}'] = np.cos(2*np.pi*k*didx/365.25)
    
    for m in ['Call Volume','CCT','Abandon Rate']:
        if m not in df.columns: continue
        f[f'{m}_l7'] = df[m].shift(7)
        f[f'{m}_l14'] = df[m].shift(14)
        f[f'{m}_l28'] = df[m].shift(28)
        f[f'{m}_l365'] = df[m].shift(365)
        f[f'{m}_r7'] = df[m].rolling(7).mean()
        f[f'{m}_r14'] = df[m].rolling(14).mean()
        f[f'{m}_r30'] = df[m].rolling(30).mean()
        f[f'{m}_sd7'] = df[m].rolling(7).std()
        f[f'{m}_ew'] = df[m].ewm(span=7).mean()
    
    sc = f'Staff_{port}'
    f = f.merge(staff[['Date',sc]].rename(columns={sc:'agents'}), on='Date', how='left')
    f['agents_d'] = f['agents'].diff()
    if 'Call Volume' in df.columns:
        cv = df[['Date','Call Volume']].rename(columns={'Call Volume':'_cv'})
        f = f.merge(cv, on='Date', how='left')
        f['cpa'] = f['_cv'] / f['agents']
        f['cpa_l7'] = f['cpa'].shift(7)
        f.drop('_cv', axis=1, inplace=True)
    
    for m in ['Call Volume','CCT','Abandon Rate']:
        if m in df.columns:
            f[f'tgt_{m}'] = df[m]
    return f

feats = {}
for p in portfolios:
    feats[p] = make_features(daily[p], p)
print({p: feats[p].shape for p in portfolios})


{'A': (731, 68), 'B': (731, 68), 'C': (731, 68), 'D': (731, 68)}


In [7]:
# intraday profiles from interval data
# splitting by high/low volume days bc the shape is different

cv_prof, abd_prof, cct_prof = {}, {}, {}

for p in portfolios:
    df = intervals[p].copy()
    df['dow'] = df['Date'].dt.dayofweek
    dtot = df.groupby('Date')['Call Volume'].sum().reset_index()
    dtot.columns = ['Date','dtot']
    df = df.merge(dtot, on='Date')
    abd_tot = df.groupby('Date')['Abandoned Calls'].transform('sum')
    df['cv_pct'] = df['Call Volume'] / df['dtot'].replace(0, np.nan)
    df['abd_pct'] = df['Abandoned Calls'] / abd_tot.replace(0, np.nan)
    
    cv_prof[p], abd_prof[p], cct_prof[p] = {}, {}, {}
    
    for dow in range(7):
        sub = df[df['dow']==dow]
        med = sub.groupby('Date')['Call Volume'].sum().median()
        cv_prof[p][dow] = {'med': med}
        abd_prof[p][dow] = {}
        cct_prof[p][dow] = {}
        
        for lvl, msk in [('hi', sub['dtot']>=med), ('lo', sub['dtot']<med)]:
            s = sub[msk]
            if len(s) < 10: s = sub
            
            for col, store in [('cv_pct',cv_prof), ('abd_pct',abd_prof)]:
                pr = s.groupby('slot')[col].median()
                a = np.zeros(48)
                a[pr.index.astype(int)] = pr.values
                a = np.nan_to_num(a, 0)
                a = gaussian_filter1d(a, sigma=0.7)
                if a.sum() > 0: a /= a.sum()
                store[p][dow][lvl] = a
            
            pr = s.groupby('slot')['CCT'].median()
            a = np.zeros(48)
            a[pr.index.astype(int)] = pr.values
            a = np.nan_to_num(a, 0)
            a = gaussian_filter1d(a, sigma=0.7)
            cct_prof[p][dow][lvl] = a

prof_cct_mean = {}
for p in portfolios:
    msk = (daily[p]['Date']>='2024-04-01') & (daily[p]['Date']<='2024-06-30')
    prof_cct_mean[p] = daily[p].loc[msk, 'CCT'].mean()

print('profiles done')


profiles done


In [8]:
# train models + predict august

def feat_cols(df):
    skip = ['Date','tgt_Call Volume','tgt_CCT','tgt_Abandon Rate']
    return [c for c in df.columns if c not in skip]

preds = {}
aug = pd.date_range('2025-08-01','2025-08-31')

for p in portfolios:
    print(f'\n--- {p} ---')
    ft = feats[p]
    cols = feat_cols(ft)
    ok = ft[cols].notna().all(axis=1)
    cl = ft[ok].copy()
    
    trn = cl['Date'] < '2025-07-01'
    val = (cl['Date']>='2025-07-01') & (cl['Date']<'2025-08-01')
    Xtr, Xv = cl.loc[trn, cols].values, cl.loc[val, cols].values
    
    preds[p] = {}
    d = daily[p]
    a24 = d[(d['Date'].dt.month==8)&(d['Date'].dt.year==2024)]
    
    for met in ['Call Volume','CCT']:
        ytr = cl.loc[trn, f'tgt_{met}'].values
        yv = cl.loc[val, f'tgt_{met}'].values
        
        # tried diff hyperparams, these worked best on july validation
        if met == 'CCT':
            gb = HistGradientBoostingRegressor(loss='quantile', quantile=0.52,
                max_iter=500, max_depth=5, learning_rate=0.04, l2_regularization=1.5,
                min_samples_leaf=15, random_state=42)
        else:
            # quantile=0.55 predicts above median (helps w/ asymmetric penalty)
            gb = HistGradientBoostingRegressor(loss='quantile', quantile=0.55,
                max_iter=500, max_depth=6, learning_rate=0.05, l2_regularization=1.0,
                min_samples_leaf=10, random_state=42)
        gb.fit(Xtr, ytr)
        
        # also tried xgboost but wasnt in the docker image
        rd = Ridge(alpha=1.0)
        rd.fit(np.nan_to_num(Xtr,0), ytr)
        
        vp = 0.6*gb.predict(Xv) + 0.4*rd.predict(np.nan_to_num(Xv,0))
        print(f'  {met} MAE: {mean_absolute_error(yv,vp):.2f}')
        
        amsk = (ft['Date']>='2025-08-01') & (ft['Date']<='2025-08-31')
        Xa = ft.loc[amsk, cols].fillna(method='ffill').fillna(method='bfill').fillna(0)
        ml = 0.6*gb.predict(Xa.values) + 0.4*rd.predict(np.nan_to_num(Xa.values,0))
        
        # baseline: last august * yoy growth
        g24 = d[(d['Date'].dt.year==2024)&(d['Date'].dt.month<=7)][met].mean()
        g25 = d[(d['Date'].dt.year==2025)&(d['Date'].dt.month<=7)][met].mean()
        gr = g25/g24 if g24>0 else 1.0
        bl = np.zeros(31)
        for i,dt in enumerate(aug):
            m = a24[a24['Date'].dt.dayofweek==dt.dayofweek][met].values
            bl[i] = m.mean()*gr if len(m)>0 else a24[met].mean()*gr
        
        if met == 'CCT':
            combo = 0.7*ml[:31] + 0.3*bl
            ast = staff[(staff['Date'].dt.month==8)&(staff['Date'].dt.year==2025)]
            if len(ast)>=31:
                ag = ast[f'Staff_{p}'].values[:31]
                cpa = bl / np.maximum(ag,1)
                sc = staff_lr[p]['cct'].predict(cpa.reshape(-1,1))
                sc = np.clip(sc, 200, 600)
                final = 0.7*combo + 0.3*sc
            else:
                final = combo
        else:
            final = 0.7*ml[:31] + 0.3*bl
        
        preds[p][met] = final
        print(f'  {met} aug avg: {final.mean():.1f}')
    
    # abandon rate - GB keeps predicting 0 bc values are tiny (1-2%)
    # just using recent historical avgs by day of week instead
    recent = d[(d['Date']>='2025-06-01')&(d['Date']<'2025-08-01')]
    abd = np.zeros(31)
    for i,dt in enumerate(aug):
        dw = dt.dayofweek
        r = recent[recent['Date'].dt.dayofweek==dw]['Abandon Rate']
        a = a24[a24['Date'].dt.dayofweek==dw]['Abandon Rate']
        if len(r)>0 and len(a)>0:
            abd[i] = 0.6*r.mean() + 0.4*a.mean()
        elif len(r)>0:
            abd[i] = r.mean()
        elif len(a)>0:
            abd[i] = a.mean()
        else:
            abd[i] = d['Abandon Rate'].tail(60).mean()
    abd *= 1.1
    abd = np.clip(abd, 0.002, 0.25)
    preds[p]['Abandon Rate'] = abd
    abd_avg = a24['Abandon Rate'].mean()
    print(f'  ABD: {abd.mean():.4f} (was {abd_avg:.4f} in aug24)')

print('done')



--- A ---


  Call Volume MAE: 127.12
  Call Volume aug avg: 3833.7


  CCT MAE: 8.21
  CCT aug avg: 317.2
  ABD: 0.0120 (was 0.0110 in aug24)

--- B ---


  Call Volume MAE: 271.85
  Call Volume aug avg: 7840.0


  CCT MAE: 9.30
  CCT aug avg: 339.6
  ABD: 0.0176 (was 0.0150 in aug24)

--- C ---


  Call Volume MAE: 377.22
  Call Volume aug avg: 18483.2


  CCT MAE: 6.64
  CCT aug avg: 328.9
  ABD: 0.0108 (was 0.0093 in aug24)

--- D ---


  Call Volume MAE: 512.22
  Call Volume aug avg: 9644.0


  CCT MAE: 5.51
  CCT aug avg: 322.7
  ABD: 0.0144 (was 0.0154 in aug24)
done


In [9]:
# disaggregate to 30min intervals
aug_dates = pd.date_range('2025-08-01','2025-08-31')
res = {p: {'cv':[],'abd':[],'ar':[],'cct':[]} for p in portfolios}

for p in portfolios:
    dcv = preds[p]['Call Volume']
    dcct = preds[p]['CCT']
    dar = preds[p]['Abandon Rate']
    dabd = dcv * dar
    
    for i,dt in enumerate(aug_dates):
        dw = dt.dayofweek
        lvl = 'hi' if dcv[i] >= cv_prof[p][dw]['med'] else 'lo'
        
        cv = dcv[i] * cv_prof[p][dw][lvl]
        ab = dabd[i] * abd_prof[p][dw][lvl]
        sc = dcct[i] / prof_cct_mean[p] if prof_cct_mean[p]>0 else 1.0
        cc = cct_prof[p][dw][lvl] * sc
        ar = np.where(cv>0, ab/cv, 0.0)
        
        res[p]['cv'].append(cv)
        res[p]['abd'].append(ab)
        res[p]['ar'].append(ar)
        res[p]['cct'].append(cc)

for p in portfolios:
    for k in res[p]:
        res[p][k] = np.array(res[p][k])

for p in portfolios:
    print(f'{p}: sum={res[p]["cv"][0].sum():.0f} vs {preds[p]["Call Volume"][0]:.0f}')


A: sum=4421 vs 4421
B: sum=8947 vs 8947
C: sum=21509 vs 21509
D: sum=10841 vs 10841


In [10]:
# bias up bc under-forecasting hurts more than over
# tuned per portfolio based on how far off we were from aug 2024
bias = {'A': 1.12, 'B': 1.20, 'C': 1.11, 'D': 1.18}

for p in portfolios:
    b = bias[p]
    res[p]['cv'] *= b
    res[p]['abd'] *= b
    res[p]['cct'] *= 1.05

# cleanup
for p in portfolios:
    res[p]['cv'] = np.clip(res[p]['cv'], 0, None)
    res[p]['abd'] = np.clip(res[p]['abd'], 0, None)
    res[p]['cct'] = np.clip(res[p]['cct'], 0, None)
    
    bad = res[p]['abd'] > res[p]['cv']
    res[p]['abd'][bad] = res[p]['cv'][bad]
    
    cv, ab = res[p]['cv'], res[p]['abd']
    res[p]['ar'] = np.clip(np.where(cv>0, ab/cv, 0.0), 0, 1)
    res[p]['cv'] = np.round(res[p]['cv']).astype(int)
    res[p]['abd'] = np.round(res[p]['abd']).astype(int)

for p in portfolios:
    print(f'{p}: {res[p]["cv"].sum():,}')


A: 133,120
B: 291,654
C: 635,993
D: 352,764


In [11]:
# write csv
rows = []
for day in range(31):
    for slot in range(48):
        h = slot // 2
        m = (slot % 2) * 30
        row = {'Month':'August', 'Day':str(day+1), 'Interval':f'{h}:{m:02d}'}
        for p in portfolios:
            row[f'Calls_Offered_{p}'] = int(res[p]['cv'][day,slot])
            row[f'Abandoned_Calls_{p}'] = int(res[p]['abd'][day,slot])
            row[f'Abandoned_Rate_{p}'] = round(float(res[p]['ar'][day,slot]), 6)
            row[f'CCT_{p}'] = round(float(res[p]['cct'][day,slot]), 2)
        rows.append(row)

sub = pd.DataFrame(rows)[template.columns.tolist()]
out = os.path.join(os.getcwd(), 'forecast_v3.csv')
sub.to_csv(out, index=False)
print(f'saved to {out}')
print(sub.shape)


saved to /Users/chideraibe/Documents/Datathon/v3/forecast_v3.csv
(1488, 19)


In [12]:
# checks
assert sub.shape == (1488, 19)
assert not sub.isnull().any().any()
for p in portfolios:
    assert (sub[f'Calls_Offered_{p}']>=0).all()
    assert (sub[f'Abandoned_Calls_{p}']>=0).all()
    assert (sub[f'Abandoned_Calls_{p}']<=sub[f'Calls_Offered_{p}']).all()
    assert (sub[f'Abandoned_Rate_{p}']<=1).all()
    assert (sub[f'CCT_{p}']>=0).all()
print('all good')

for p in portfolios:
    a24 = daily[p][(daily[p]['Date'].dt.month==8)&(daily[p]['Date'].dt.year==2024)]
    fc = sub[f'Calls_Offered_{p}'].sum()
    ac = a24['Call Volume'].sum()
    print(f'{p}: {fc:,} vs {ac:,.0f} (ratio {fc/ac:.3f})')


all good
A: 133,120 vs 127,759 (ratio 1.042)
B: 291,654 vs 286,317 (ratio 1.019)
C: 635,993 vs 621,313 (ratio 1.024)
D: 352,764 vs 347,005 (ratio 1.017)
